In [1]:
# ler excel
import pandas as pd

# folha Resumo
df_smh = pd.read_excel('diagnostico.xlsx', sheet_name='Resumo')
# Exibir as primeiras linhas do DataFrame
print(df_smh.head())

  Código do diagnóstico                                        Diagnóstico  \
0                  E785                        Hyperlipidemia, Unspecified   
1                   I10                   Essential (Primary) Hypertension   
2                 Z7984  Long Term (Current) Use Of Oral Hypoglycemic D...   
3                  E119     Type 2 Diabetes Mellitus Without Complications   
4                  E669                               Obesity, Unspecified   

   Qtd de episódios  
0              1599  
1              1580  
2               591  
3               538  
4               491  


In [2]:
df_icd10 = pd.read_excel('icd10cm.xlsx', sheet_name='ICD10CM')
print(df_icd10.head())

  Capitulo ICD-10-CM_ Código  \
0                    A00-B99   
1                    A00-B99   
2                    A00-B99   
3                    A00-B99   
4                    A00-B99   

                             Capitulo ICD-10-CM_desc  \
0  Certain infectious and parasitic diseases (A00...   
1  Certain infectious and parasitic diseases (A00...   
2  Certain infectious and parasitic diseases (A00...   
3  Certain infectious and parasitic diseases (A00...   
4  Certain infectious and parasitic diseases (A00...   

                   Capitulo ICD-10-CM_desc_PT Secção ICD-10-CM_Código  \
0  Algumas doenças infecciosas e parasitárias                 A00-A09   
1  Algumas doenças infecciosas e parasitárias                 A00-A09   
2  Algumas doenças infecciosas e parasitárias                 A00-A09   
3  Algumas doenças infecciosas e parasitárias                 A00-A09   
4  Algumas doenças infecciosas e parasitárias                 A00-A09   

            Secção ICD-10-CM_De

In [3]:
# filtrar df_id10 coluna Código para corresponder à coluna Diagnóstico em df_smh
filtered_icd10 = df_icd10[df_icd10['Código'].isin(df_smh['Código do diagnóstico'])]
print(len(filtered_icd10))

2097


In [4]:

# guardar 'Descrição PT_(Longa)', 'Capitulo ICD-10-CM_desc',      'Descrição PT_(Curta)','Código', 'Válido',
bd = filtered_icd10[['Código', 'Capitulo ICD-10-CM_desc_PT', 'Descrição PT_(Longa)','Secção ICD-10-CM_Desc_PT', 'Válido']]

# guardar em excel
print(len(bd))


2097


In [5]:
#  'Capitulo ICD-10-CM_desc_PT'  unique
capitulos = bd['Capitulo ICD-10-CM_desc_PT'].unique()

print(capitulos)

sections = bd['Secção ICD-10-CM_Desc_PT'].unique()

['Algumas doenças infecciosas e parasitárias' 'Neoplasias'
 'Doenças do sangue e dos órgãos hematopoéticos e alguns transtornos imunitários'
 'Doenças endócrinas, nutricionais e metabólicas'
 'Transtornos mentais, comportamentais e de neurodesenvolvimento'
 'Doenças do sistema nervoso' 'Doenças do olho e anexos'
 'Doenças do ouvido e da apófise mastóide'
 'Doenças do aparelho circulatório' 'Doenças do aparelho respiratório'
 'Doenças do aparelho digestivo' 'Doenças da pele e do tecido subcutâneo'
 'Doenças do aparelho osteomuscular e do tecido conjuntivo'
 'Doenças do aparelho geniturinário'
 'Malformações congénitas, deformações, anomalias cromossómicas, e doenças genéticas (Q00-QA0)'
 'Sintomas/sinais/achados anormais d exames clínicos/laboratoriais,ÑClassificadosNoutraParte'
 'Lesões, envenenamento e algumas outras consequências de causas externas'
 'Códigos para fins especiais (U00-U85)' 'Causas externas de morbilidade'
 'Fatores que influenciam o estado de saúde e o contacto com o

In [6]:
# add sections to database
import sqlite3
con = sqlite3.connect('../database/database.sqlite')



for data in sections:
    con.execute('INSERT INTO sections (name, kind) VALUES (?, ?)', (data, "diagnosticos"))
con.commit()

con.close()

In [7]:
# guardar na base de dados
import sqlite3
con = sqlite3.connect('../database/database.sqlite')


for data in capitulos:
    con.execute('INSERT INTO categories (name, kind) VALUES (?, ?)', (data, "diagnosticos"))
    
con.commit()

In [8]:

# Capitulo ICD-10-CM_desc_PT as categoria
bd = bd.rename(columns={'Capitulo ICD-10-CM_desc_PT': 'categoria', 'Secção ICD-10-CM_Desc_PT': 'secao', 'Código': 'codigo', 'Descrição PT_(Longa)': 'description', 'Válido': 'valid'})    
print(bd.head())



    codigo                                   categoria  \
47   A0471  Algumas doenças infecciosas e parasitárias   
48   A0472  Algumas doenças infecciosas e parasitárias   
94     A09  Algumas doenças infecciosas e parasitárias   
96    A150  Algumas doenças infecciosas e parasitárias   
128   A182  Algumas doenças infecciosas e parasitárias   

                                           description  \
47   Enterocolite devido a Clostridium Difficile, r...   
48   Enterocolite devido a Clostridium Difficile, n...   
94   Gastroenterite e colite infeciosa, sem outra e...   
96                                Tuberculose pulmonar   
128              Linfadenopatia tuberculosa periférica   

                               secao  valid  
47   Doenças infecciosas intestinais      1  
48   Doenças infecciosas intestinais      1  
94   Doenças infecciosas intestinais      1  
96                       Tuberculose      1  
128                      Tuberculose      1  


In [9]:
import sqlite3

con = sqlite3.connect('../database/database.sqlite', timeout=30)
cur = con.cursor()

# Evita problemas de locked
cur.execute("PRAGMA journal_mode=WAL;")
cur.execute("PRAGMA busy_timeout = 30000;")
cur.execute("PRAGMA synchronous = NORMAL;")

# Cache para evitar SELECT duplicado
cache_categoria = {}
cache_codigo = {}
cache_secao = {}

# Specialty carregada uma vez
specialty_id = cur.execute(
    "SELECT id FROM specialties WHERE name = ?",
    ("Cirurgia Geral",)
).fetchone()
specialty_id = specialty_id[0] if specialty_id else None

# Lista para inserção em massa
insercoes = []

# Supondo que o DataFrame seja df_smh ou df_diagnosticos
for row in bd.itertuples():

    # ---------- Categoria (cache) ----------
    categoria = row.categoria
    if categoria not in cache_categoria:
        r = cur.execute(
            "SELECT id FROM categories WHERE name = ?",
            (categoria,)
        ).fetchone()
        cache_categoria[categoria] = r[0] if r else None
    categoria_id = cache_categoria[categoria]

    # ---------- Código ICD (cache) ----------
    codigo = row.codigo
    if codigo not in cache_codigo:
        r = cur.execute(
            "SELECT id FROM icd10cms WHERE codigo = ?",
            (codigo,)
        ).fetchone()
        cache_codigo[codigo] = r[0] if r else None
    codigo_id = cache_codigo[codigo]

    # ---------- Seção (se existir) ----------
    secao = row.secao if hasattr(row, "secao") else None
    if secao:
        if secao not in cache_secao:
            r = cur.execute(
                "SELECT id FROM sections WHERE name = ?",
                (secao,)
            ).fetchone()
            cache_secao[secao] = r[0] if r else None
        secao_id = cache_secao[secao]
    else:
        secao_id = None

    
    # Preparar registro para executemany
    insercoes.append((
        "icd10cms",
        codigo_id,
        categoria_id,
        specialty_id,
        secao_id
        
    ))

# -------- INSERÇÃO EM MASSA ----------
cur.executemany("""
    INSERT INTO favoritos (
        tabela_origem, codigo_id, category_id, specialty_id, section_id
    )
    VALUES (?, ?, ?, ?, ?)
""", insercoes)

con.commit()
con.close()
